# 数据集准备 (Data Preparation)

本 Notebook 负责拉取并清洗两套训练语料至 `train/dataset/`：
- **预训练语料**：MiniMind `pretrain_t2t_mini`（通用纯文本）
- **SFT 对话语料**：Nana-catgirl-110k（单轮猫娘人设对话）

清洗流程会自动过滤思维链 (CoT)、颜文字/Emoji、System 提示词与工具调用标记，并剔除预留给结构信号的 C0 控制字节，输出统一规范为单字段 `{"text": ...}`。

## 快速上手

代码支持断点续传与幂等执行，直接从上往下顺序运行 **4 个 Cell** 即可。全局控制项集中在 **第 1 格**：

| 配置项 | 默认值 | 说明 |
| :--- | :--- | :--- |
| `USE_PROXY` / `PROXY_URL` | `True` / `127.0.0.1:7890` | 本地代理配置（国内下载 HF / ModelScope 时使用；不使用设为 `False`） |
| `USE_HF_MIRROR` | `True` (`hf-mirror.com`) | 启用 HuggingFace 国内镜像加速 |
| `PRETRAIN_MIN/MAX_BYTES` | `0 / None` | 预训练文本字节长度阈值过滤（`None` 为不限制） |
| `NANA_MIN/MAX_BYTES` | `0 / None` | 对话文本字节长度阈值过滤（`None` 为不限制） |
| `DELETE_INTERMEDIATES` | `False` | 第 4 格是否清理原始下载文件与缓存（建议确认数据无误后设为 `True`） |

### 执行步骤说明
1. **第 1 格（配置与自检）**：校验 Python 依赖、磁盘可用空间（需 ≥ 4GB）并注入代理与镜像环境变量。
2. **第 2 格（下载原始语料）**：拉取 ModelScope 与 HuggingFace 语料。若本地已存在完整文件或清洗产物，将自动跳过。
3. **第 3 格（语料清洗标准化）**：执行正则清洗与格式转换，同时剔除预留的 C0 控制字节（只留 `\t` / `\n`），保证 `<ETX>`（0x03）只作为序列结束标记出现。系统通过 `*.meta.json` 记录清洗参数指纹，参数未变动时自动跳过；若需强制重洗，删除对应的 `*_clean.jsonl` 即可。
4. **第 4 格（体检与清理）**：统计清洗前后的字节长度分位数与保留比例，打印样本预览。确认无误后可开启 `DELETE_INTERMEDIATES = True` 释放约 1.4GB 磁盘空间。

## 数据与产物清单

| 阶段 | 文件名 | 来源 / 格式 | 说明 |
| :--- | :--- | :--- | :--- |
| 原始语料 | `pretrain_t2t_mini.jsonl` | ModelScope (`gongjy/minimind_dataset`) | 预训练原始语料 (~1.24 GB) |
| 原始语料 | `nana_catgirl_dataset_110k.jsonl` | HuggingFace (`cyberlangke/Nana-catgirl-dataset-110k`) | 对话原始语料 (~216 MB) |
| 清洗成品 | `pretrain_clean.jsonl` | JSONL (`{"text": "..."}`) | 预训练清洗产物，纯文本字段 |
| 清洗成品 | `nana_clean.jsonl` | JSONL (`{"text": "user:...\nbot:..."}`) | 对话微调产物，单轮规范格式；序列结束的 `<ETX>` 由打包脚本补 |


In [1]:
# ============================================================
# 1. 配置与自检
# 本格集中全部可调参数，同时检查 Python 依赖与磁盘空间，并把代理、镜像写入环境变量，
# 供后面的下载步骤使用。
# ============================================================

from pathlib import Path

# ---- 目录：定位 train/dataset/，不依赖 notebook 的启动位置 ----
_cwd = Path.cwd()
if _cwd.name == "notebooks":
    TRAIN_DIR = _cwd.parent
elif (_cwd / "dataset").is_dir():
    TRAIN_DIR = _cwd
else:
    TRAIN_DIR = _cwd / "train"
DATASET_DIR = TRAIN_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# ---- 代理：本机 Clash 默认监听 127.0.0.1:7890，直连时请关掉 ----
USE_PROXY = True
PROXY_URL = "http://127.0.0.1:7890"

# ---- HuggingFace 镜像：国内直连 huggingface.co 通常不通，改走 hf-mirror.com ----
USE_HF_MIRROR = True
HF_MIRROR = "https://hf-mirror.com"

# ---- 数据源：仓库 ID 与文件名，EXPECT_BYTES 用来校验下载是否完整 ----
MS_DATASET_ID = "gongjy/minimind_dataset"
MS_FILE = "pretrain_t2t_mini.jsonl"
MS_EXPECT_BYTES = 1_241_043_656

HF_REPO_ID = "cyberlangke/Nana-catgirl-dataset-110k"
HF_FILE = "nana_catgirl_dataset_110k.jsonl"
HF_EXPECT_BYTES = 216_781_504

# ---- 路径：原始语料与清洗产物的落盘位置 ----
RAW_PRETRAIN = DATASET_DIR / MS_FILE
RAW_NANA = DATASET_DIR / HF_FILE
CLEAN_PRETRAIN = DATASET_DIR / "pretrain_clean.jsonl"
CLEAN_NANA = DATASET_DIR / "nana_clean.jsonl"

# ---- 对话模板：拼成 user:<内容> 换行 bot:<内容>，冒号后不加空格 ----
USER_TAG = "user:"
BOT_TAG = "bot:"

# ---- 长度过滤：单位是 UTF-8 字节数，0 或 None 表示不限制 ----
PRETRAIN_MIN_BYTES = 0
PRETRAIN_MAX_BYTES = None
NANA_MIN_BYTES = 0
NANA_MAX_BYTES = None

# ---- 中间产物清理开关：默认关闭，确认清洗结果无误后再打开 ----
DELETE_INTERMEDIATES = False

print("TRAIN_DIR   :", TRAIN_DIR)
print("DATASET_DIR :", DATASET_DIR)
print("代理        :", PROXY_URL if USE_PROXY else "不使用")
print("HF 镜像     :", HF_MIRROR if USE_HF_MIRROR else "不使用")

# ------------------------------------------------------------
# 工具函数：完整性校验、长度统计与原子写入
# ------------------------------------------------------------
import json, os, shutil, time


def human(n):
    """把字节数换算成便于阅读的字符串，例如 1.20 GB。"""
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return "%.2f %s" % (n, unit)
        n /= 1024.0


def size_of(path):
    """返回文件大小，文件不存在时返回 0。"""
    p = Path(path)
    return p.stat().st_size if p.is_file() else 0


def file_ok(path, expect_bytes=None, tol=0.02):
    """判断文件是否完整：存在、非空，并且在给定期望大小时体积落在容差范围内。"""
    p = Path(path)
    if not p.is_file():
        return False
    size = p.stat().st_size
    if size == 0:
        return False
    if expect_bytes is not None and abs(size - expect_bytes) > expect_bytes * tol:
        return False
    return True


def byte_len(s):
    """返回字符串的 UTF-8 字节长度，与字节级 tokenizer 的 token 数一致。"""
    return len(s.encode("utf-8"))


def write_jsonl_atomic(path, rows):
    """逐行写出 JSONL：先写临时文件再原子替换，中断时只会留下 .tmp，不会产生半截成品。"""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    n = 0
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            n += 1
    os.replace(tmp, path)
    return n


def meta_path(out_path):
    """返回清洗产物对应的 meta 文件路径。"""
    return Path(str(out_path) + ".meta.json")


def write_meta(out_path, params):
    """把本次清洗使用的参数写入 meta 文件，供下次运行时判断产物是否过期。"""
    meta_path(out_path).write_text(
        json.dumps(params, ensure_ascii=False, indent=2, sort_keys=True),
        encoding="utf-8")


def meta_matches(out_path, params):
    """判断产物能否直接复用：文件存在且 meta 参数一致时返回 True，参数变化时返回 False 触发重跑。"""
    mp = meta_path(out_path)
    if not mp.is_file():
        return False
    try:
        return json.loads(mp.read_text(encoding="utf-8")) == params
    except Exception:
        return False


def percentile(sorted_vals, q):
    """在已排序列表上取 q 分位数，空列表返回 0。"""
    if not sorted_vals:
        return 0
    idx = min(len(sorted_vals) - 1, int(round(q * (len(sorted_vals) - 1))))
    return sorted_vals[idx]


def report_lengths(lens, label):
    """打印字节长度的分位数，以及若干候选阈值下的样本保留比例。"""
    if not lens:
        print("  %s: 没有数据" % label)
        return
    lens = sorted(lens)
    print("  %s: n=%d" % (label, len(lens)))
    print("    长度  min=%d  p25=%d  p50=%d  p75=%d  p90=%d  p95=%d  p99=%d  max=%d"
          % tuple([percentile(lens, q) for q in (0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0)]))
    parts = []
    for th in (128, 256, 384, 512, 768, 1024, 2048, 4096):
        keep = sum(1 for v in lens if v <= th)
        parts.append("<=%d:%.1f%%" % (th, 100.0 * keep / len(lens)))
    print("    保留比例 " + "  ".join(parts))


# ------------------------------------------------------------
# 自检：检查依赖与磁盘空间，并注入代理、镜像环境变量
# ------------------------------------------------------------
import importlib, platform, sys

print()
print("Python   :", sys.version.split()[0])
print("平台     :", platform.platform())
print("解释器   :", sys.executable)

need = ["modelscope", "huggingface_hub", "requests"]
missing = []
for mod in need:
    try:
        m = importlib.import_module(mod)
        print("OK   %-18s %s" % (mod, getattr(m, "__version__", "?")))
    except ImportError:
        print("缺失 %-18s" % mod)
        missing.append(mod)
if missing:
    raise RuntimeError("缺少依赖 %s，请先执行：python -m pip install %s"
                       % (missing, " ".join(missing)))

free = shutil.disk_usage(str(TRAIN_DIR)).free
print("磁盘剩余 :", human(free))
if free < 4 * 1024 ** 3:
    print("警告：剩余空间不足 4GB，而下载与清洗大约需要 3GB 余量。")

# 代理与镜像必须在这里写入环境变量，后面的下载步骤会直接复用。
if USE_PROXY:
    os.environ["HTTP_PROXY"] = PROXY_URL
    os.environ["HTTPS_PROXY"] = PROXY_URL
    print("已设置代理   :", PROXY_URL)
else:
    for k in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
        os.environ.pop(k, None)
    print("未使用代理")

if USE_HF_MIRROR:
    os.environ["HF_ENDPOINT"] = HF_MIRROR
    print("HF_ENDPOINT  :", HF_MIRROR)
else:
    os.environ.pop("HF_ENDPOINT", None)
    print("HF_ENDPOINT  : 默认（huggingface.co）")


TRAIN_DIR   : D:\Git_Repository\nanomeow\train
DATASET_DIR : D:\Git_Repository\nanomeow\train\dataset
代理        : http://127.0.0.1:7890
HF 镜像     : https://hf-mirror.com

Python   : 3.11.9
平台     : Windows-10-10.0.22631-SP0
解释器   : D:\Git_Repository\nanomeow\.venv\Scripts\python.exe


OK   modelscope         1.40.1
OK   huggingface_hub    1.33.0
OK   requests           2.34.2
磁盘剩余 : 632.63 GB
已设置代理   : http://127.0.0.1:7890
HF_ENDPOINT  : https://hf-mirror.com


D:\Git_Repository\nanomeow\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. 下载原始语料（ModelScope + HuggingFace）
# 本格先探测两个数据源是否连通，再把原始语料下载到 train/dataset/ 下；
# 文件已完整或清洗产物已存在时自动跳过。
# ============================================================

import requests


def probe(name, url):
    """探测 URL 是否可达：只要能拿到任意 HTTP 状态码，就认为线路是通的。"""
    try:
        r = requests.head(url, timeout=15, allow_redirects=True)
        print("  %-12s HTTP %-4s %s" % (name, r.status_code, url))
        return True
    except Exception as e:
        print("  %-12s 失败 %-6s %s" % (name, type(e).__name__, url))
        return False


print("=" * 60)
print("步骤 2：下载原始语料")
print("=" * 60)

print("连通性：")
ok_ms = probe("modelscope", "https://www.modelscope.cn")
ok_hf = probe("huggingface", HF_MIRROR if USE_HF_MIRROR else "https://huggingface.co")
if not ok_ms and not ok_hf:
    raise RuntimeError("两个数据源都不通，请检查 USE_PROXY / PROXY_URL")
if not ok_ms:
    print("提示：ModelScope 不通，预训练语料将无法下载。")
if not ok_hf:
    print("提示：HuggingFace 不通，可以试试打开 USE_HF_MIRROR。")

print()
print("预训练语料 %s / %s" % (MS_DATASET_ID, MS_FILE))
if CLEAN_PRETRAIN.exists():
    print("  清洗产物已存在，跳过下载。")
elif file_ok(RAW_PRETRAIN, MS_EXPECT_BYTES):
    print("  文件已完整，跳过下载（%s）。" % human(size_of(RAW_PRETRAIN)))
else:
    from modelscope.hub.snapshot_download import snapshot_download
    t0 = time.time()
    snapshot_download(repo_id=MS_DATASET_ID, repo_type="dataset",
                      allow_patterns=[MS_FILE], local_dir=str(DATASET_DIR))
    if not file_ok(RAW_PRETRAIN, MS_EXPECT_BYTES):
        raise RuntimeError("下载后校验失败：%s 的实际大小是 %s" % (RAW_PRETRAIN, size_of(RAW_PRETRAIN)))
    print("  下载完成，用时 %.1f 秒，大小 %s。" % (time.time() - t0, human(size_of(RAW_PRETRAIN))))

print()
print("对话语料 %s / %s" % (HF_REPO_ID, HF_FILE))
if CLEAN_NANA.exists():
    print("  清洗产物已存在，跳过下载。")
elif file_ok(RAW_NANA, HF_EXPECT_BYTES):
    print("  文件已完整，跳过下载（%s）。" % human(size_of(RAW_NANA)))
else:
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILE,
                           repo_type="dataset", local_dir=str(DATASET_DIR))
    if not file_ok(RAW_NANA, HF_EXPECT_BYTES):
        raise RuntimeError("下载后校验失败：%s 的实际大小是 %s" % (RAW_NANA, size_of(RAW_NANA)))
    print("  下载完成，用时 %.1f 秒，落盘位置 %s。" % (time.time() - t0, path))


步骤 2：下载原始语料
连通性：


  modelscope   HTTP 200  https://www.modelscope.cn


  huggingface  HTTP 200  https://hf-mirror.com

预训练语料 gongjy/minimind_dataset / pretrain_t2t_mini.jsonl
  清洗产物已存在，跳过下载。

对话语料 cyberlangke/Nana-catgirl-dataset-110k / nana_catgirl_dataset_110k.jsonl
  清洗产物已存在，跳过下载。


In [3]:
# ============================================================
# 3. 清洗语料，产出 nana_clean.jsonl 与 pretrain_clean.jsonl
# 本格先定义清洗规则并自检，再依次清洗对话语料和预训练语料；
# 清洗参数会写入 meta 文件，参数未变化时跳过重跑。
# ============================================================

import re

# 规则 1：括号块内部若不含中文、ASCII 字母或数字，就整块当作颜文字删掉；
#   因此 (2024)、（附录 A）这类正常引用会被保留。
_OPEN = "([{（［｛【〔〈《₍﹁﹃"
_CLOSE = ")]}）］｝】〕〉》₎﹂﹄"
_RESIDUE = "∫و"
_BLOCK = re.compile("[" + re.escape(_RESIDUE) + "]?[" + re.escape(_OPEN) + "]([^"
                    + re.escape(_OPEN + _CLOSE) + "\\n]{0,24}?)[" + re.escape(_CLOSE)
                    + "][" + re.escape(_RESIDUE) + "]?")
_KEEP_INNER = re.compile(r"[\u4e00-\u9fffA-Za-z0-9]")

# 规则 2：删除 Emoji 与装饰符号区段。
_SYMBOL = re.compile("[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U00002B00-\U00002BFF"
                     "\U0000FE00-\U0000FE0F\U000020E3\U00003030\U0000303D]")

# 规则 3：删除只用于颜文字描边的字符。这里刻意不含 × ² √ ÷ π ° → ① ∫ 等正常数学符号，
#   以免误伤正文。
_STROKE = set("╸╹╻╺╼╾╽╿⟆⟅₍₎𖥦˵˶ˋˊ˕˖˗⸝⸜▵▿▴▾◂▸◞◟◜◝◠◡﹃﹄﹁﹂⌂ヮಠฅㅂ๑ꇴʚɞ")

# 规则 4：清掉行首行尾残留的颜文字收尾字符，正文里的积分号 ∫ 不受影响。
_EDGE = "∫و"


# 规则 5：剔除预留给结构信号的 C0 控制字节。
#   C0 段（0x00-0x1F）整体预留给结构语义，本项目只用到 <ETX>（0x03）作序列结束，
#   由打包脚本在样本末尾补。正文里若自然出现 0x03，「序列结束」这个信号就不可靠了，
#   所以除空白类（\t 09 / \n 0A / \r 0D）以外的 C0 字节全部删掉，CRLF 归一成 LF。
_C0_RESERVED = "".join(chr(c) for c in range(0x20) if c not in (0x09, 0x0A, 0x0D))
_C0_RE = re.compile("[" + re.escape(_C0_RESERVED) + "]")


def _strip_reserved_c0(text):
    """删掉预留的 C0 控制字节并归一换行；输入为 str，输出为 str。"""
    t = text.replace("\r\n", "\n").replace("\r", "\n")
    return _C0_RE.sub("", t)


def _block_sub(m):
    """括号块内部含有正常文字时原样保留，否则返回空串，相当于整块删除。"""
    return m.group(0) if _KEEP_INNER.search(m.group(1)) else ""


def clean_text(text):
    """清洗颜文字与 Emoji、剔除预留的 C0 控制字节；输入为 None 时返回空字符串。"""
    if not text:
        return ""
    t = _strip_reserved_c0(text)
    t = _BLOCK.sub(_block_sub, t)
    t = _SYMBOL.sub("", t)
    t = "".join(ch for ch in t if ch not in _STROKE)
    t = "\n".join(ln.strip().strip(_EDGE).strip() for ln in t.split("\n"))
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


_SELFTEST = [
    ("主人好厉害喵₍^ > ▵ < ^₎⟆∫", "主人好厉害喵"),
    ("答案是 3×10² 哦 (๑•̀ㅂ•́)و✧", "答案是 3×10² 哦"),
    ("积分 ∫x dx = x²/2 是基础", "积分 ∫x dx = x²/2 是基础"),
    ("参考 (2024) 年的报告", "参考 (2024) 年的报告"),
    ("见（附录 A）", "见（附录 A）"),
    ("好的喵 (＾▽＾) ✨", "好的喵"),
    ("正文混进\x03控制字节", "正文混进控制字节"),
    ("换行\r\n要归一", "换行\n要归一"),
]
print("清洗规则自检：")
_all_ok = True
for src, want in _SELFTEST:
    got = clean_text(src)
    ok = got == want
    _all_ok = _all_ok and ok
    print("  %s  %-28s -> %r" % ("OK  " if ok else "FAIL", src, got))
print("自检：", "通过" if _all_ok else "有不符项")

# ------------------------------------------------------------
# 3.1 对话语料：只保留单轮 user/assistant，丢弃思维链、system 与工具调用
# ------------------------------------------------------------
NANA_PARAMS = {
    "source": HF_REPO_ID + "/" + HF_FILE,
    "user_tag": USER_TAG,
    "bot_tag": BOT_TAG,
    "min_bytes": NANA_MIN_BYTES,
    "max_bytes": NANA_MAX_BYTES,
    "drop_cot": True,
    "drop_kaomoji": True,
    "strip_reserved_c0": True,
    "drop_system": True,
    "single_turn_only": True,
}

print()
print("=" * 60)
print("步骤 3.1：清洗对话语料 ->", CLEAN_NANA.name)
print("=" * 60)

if file_ok(CLEAN_NANA) and meta_matches(CLEAN_NANA, NANA_PARAMS):
    print("产物已存在且参数未变，跳过清洗（%s）。" % human(size_of(CLEAN_NANA)))
elif not RAW_NANA.is_file():
    print("原始文件不存在（可能已被清理），无法清洗。请先跑步骤 2。")
else:
    if CLEAN_NANA.exists():
        print("产物存在但参数变了，重新清洗。")

    stats = dict(total=0, kept=0, bad_json=0, has_tools=0, bad_role=0,
                 none_content=0, empty_after_clean=0, too_short=0, too_long=0, had_cot=0)

    def nana_rows():
        with open(RAW_NANA, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                stats["total"] += 1
                try:
                    obj = json.loads(line)
                except Exception:
                    stats["bad_json"] += 1
                    continue

                if obj.get("tools"):
                    stats["has_tools"] += 1
                    continue

                msgs = obj.get("messages") or []
                if any((m.get("content") is None) for m in msgs):
                    stats["none_content"] += 1
                if any((m.get("reasoning_content") for m in msgs)):
                    stats["had_cot"] += 1

                u = [m for m in msgs if m.get("role") == "user"]
                a = [m for m in msgs if m.get("role") == "assistant"]
                if len(u) != 1 or len(a) != 1:
                    stats["bad_role"] += 1
                    continue

                uc = clean_text(u[0].get("content") or "")
                ac = clean_text(a[0].get("content") or "")
                if not uc or not ac:
                    stats["empty_after_clean"] += 1
                    continue

                text = "%s%s\n%s%s" % (USER_TAG, uc, BOT_TAG, ac)
                nb = byte_len(text)
                if NANA_MIN_BYTES and nb < NANA_MIN_BYTES:
                    stats["too_short"] += 1
                    continue
                if NANA_MAX_BYTES and nb > NANA_MAX_BYTES:
                    stats["too_long"] += 1
                    continue

                stats["kept"] += 1
                yield {"text": text}

    t0 = time.time()
    n = write_jsonl_atomic(CLEAN_NANA, nana_rows())
    write_meta(CLEAN_NANA, NANA_PARAMS)
    print("写入 %d 条，用时 %.1f 秒，大小 %s。" % (n, time.time() - t0, human(size_of(CLEAN_NANA))))
    print("原始 %d / 保留 %d / 解析失败 %d / 含工具调用 %d / 非单轮 %d / 清洗后为空 %d / 太短 %d / 太长 %d / 原始带思维链 %d"
          % (stats["total"], stats["kept"], stats["bad_json"], stats["has_tools"],
             stats["bad_role"], stats["empty_after_clean"], stats["too_short"],
             stats["too_long"], stats["had_cot"]))

# ------------------------------------------------------------
# 3.2 预训练语料：不做长度剔除，全部保留
# ------------------------------------------------------------
PRETRAIN_PARAMS = {
    "source": MS_DATASET_ID + "/" + MS_FILE,
    "min_bytes": PRETRAIN_MIN_BYTES,
    "max_bytes": PRETRAIN_MAX_BYTES,
    "strip_reserved_c0": True,
}

print()
print("=" * 60)
print("步骤 3.2：清洗预训练语料 ->", CLEAN_PRETRAIN.name)
print("=" * 60)

if file_ok(CLEAN_PRETRAIN) and meta_matches(CLEAN_PRETRAIN, PRETRAIN_PARAMS):
    print("产物已存在且参数未变，跳过清洗（%s）。" % human(size_of(CLEAN_PRETRAIN)))
elif not RAW_PRETRAIN.is_file():
    print("原始文件不存在（可能已被清理），无法清洗。请先跑步骤 2。")
else:
    if CLEAN_PRETRAIN.exists():
        print("产物存在但参数变了，重新清洗。")

    stats = dict(total=0, kept=0, bad_json=0, empty=0, too_short=0, too_long=0)

    def pretrain_rows():
        with open(RAW_PRETRAIN, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                stats["total"] += 1
                try:
                    obj = json.loads(line)
                except Exception:
                    stats["bad_json"] += 1
                    continue
                text = _strip_reserved_c0(obj.get("text") or "").strip()
                if not text:
                    stats["empty"] += 1
                    continue
                nb = byte_len(text)
                if PRETRAIN_MIN_BYTES and nb < PRETRAIN_MIN_BYTES:
                    stats["too_short"] += 1
                    continue
                if PRETRAIN_MAX_BYTES and nb > PRETRAIN_MAX_BYTES:
                    stats["too_long"] += 1
                    continue
                stats["kept"] += 1
                yield {"text": text}

    t0 = time.time()
    n = write_jsonl_atomic(CLEAN_PRETRAIN, pretrain_rows())
    write_meta(CLEAN_PRETRAIN, PRETRAIN_PARAMS)
    print("写入 %d 条，用时 %.1f 秒，大小 %s。" % (n, time.time() - t0, human(size_of(CLEAN_PRETRAIN))))
    print("原始 %d / 保留 %d / 解析失败 %d / 空文本 %d / 太短 %d / 太长 %d"
          % (stats["total"], stats["kept"], stats["bad_json"], stats["empty"],
             stats["too_short"], stats["too_long"]))


清洗规则自检：
  OK    主人好厉害喵₍^ > ▵ < ^₎⟆∫          -> '主人好厉害喵'
  OK    答案是 3×10² 哦 (๑•̀ㅂ•́)و✧       -> '答案是 3×10² 哦'
  OK    积分 ∫x dx = x²/2 是基础          -> '积分 ∫x dx = x²/2 是基础'
  OK    参考 (2024) 年的报告               -> '参考 (2024) 年的报告'
  OK    见（附录 A）                      -> '见（附录 A）'
  OK    好的喵 (＾▽＾) ✨                  -> '好的喵'
自检： 通过

步骤 3.1：清洗对话语料 -> nana_clean.jsonl
产物已存在且参数未变，跳过清洗（46.22 MB）。

步骤 3.2：清洗预训练语料 -> pretrain_clean.jsonl
产物已存在且参数未变，跳过清洗（1.16 GB）。


In [4]:
# ============================================================
# 4. 体检与清理
# 本格扫描原始语料与清洗产物，打印长度分布、保留比例和样本预览；
# 确认结果无误后，可以把第 1 格的开关打开，清理中间产物。
# ============================================================


def scan_raw(path, label, templated, limit=None):
    """逐行扫描原始 JSONL，返回总行数、解析失败条数与每条文本的字节长度列表。"""
    if not Path(path).is_file():
        print("  %s: 文件不存在，跳过。" % label)
        return None
    n = bad = 0
    lens = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            n += 1
            if limit is not None and n > limit:
                break
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue
            if templated:
                msgs = obj.get("messages") or []
                u = next((m.get("content") or "" for m in msgs if m.get("role") == "user"), "")
                a = next((m.get("content") or "" for m in msgs if m.get("role") == "assistant"), "")
                t = "%s%s\n%s%s" % (USER_TAG, u, BOT_TAG, a)
            else:
                t = obj.get("text") or ""
            lens.append(byte_len(t))
    print("  %s: 行数=%d  解析失败=%d" % (label, n, bad))
    report_lengths(lens, label)
    return n, bad, lens


def scan_clean(path, label):
    """逐行扫描清洗产物，打印字节长度分布，并统计残留的预留 C0 控制字节。"""
    if not file_ok(path):
        print("  %s: 文件不存在，跳过。" % label)
        return
    n = 0
    bad_c0 = 0
    lens = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            text = obj.get("text") or ""
            if _C0_RE.search(text):
                bad_c0 += 1
            lens.append(byte_len(text))
            n += 1
    print("  %s: n=%d  总字节=%s  含预留 C0 的行=%d（应为 0）"
          % (label, n, human(sum(lens)), bad_c0))
    report_lengths(lens, label)


print("=" * 60)
print("步骤 4：体检与清理")
print("=" * 60)

print("原始语料（全量扫描，比较慢）：")
scan_raw(RAW_PRETRAIN, "pretrain(raw)", templated=False)
scan_raw(RAW_NANA, "nana(raw)", templated=True)

print()
print("清洗产物：")
scan_clean(CLEAN_NANA, "nana(clean)")
scan_clean(CLEAN_PRETRAIN, "pretrain(clean)")

print()
print("总览：")
for p in (RAW_PRETRAIN, RAW_NANA, CLEAN_PRETRAIN, CLEAN_NANA):
    print("  [%s] %-34s %s" % ("有" if p.is_file() else "无", p.name,
                               human(size_of(p)) if p.is_file() else "-"))

print()
print("样本：")
for p in (CLEAN_NANA, CLEAN_PRETRAIN):
    if file_ok(p):
        with open(p, "r", encoding="utf-8") as f:
            sample = json.loads(f.readline())["text"]
        print("  --- %s ---" % p.name)
        print("  " + sample[:200].replace("\n", "\\n"))

# ---- 清理中间产物：原始下载文件与 HuggingFace 下载缓存 ----
print()
print("清理中间产物：")
intermediates = [RAW_PRETRAIN, RAW_NANA, DATASET_DIR / ".cache"]
if not DELETE_INTERMEDIATES:
    print("  开关为 False，跳过清理。")
else:
    if not (file_ok(CLEAN_PRETRAIN) and file_ok(CLEAN_NANA)):
        raise RuntimeError("清洗产物不完整，拒绝删除中间产物。")
    freed = 0
    for p in intermediates:
        if p.is_dir():
            sz = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
            shutil.rmtree(p)
            freed += sz
            print("  已删除目录 %s（%s）。" % (p.name, human(sz)))
        elif p.is_file():
            sz = p.stat().st_size
            p.unlink()
            freed += sz
            print("  已删除文件 %s（%s）。" % (p.name, human(sz)))
        else:
            print("  %s 不存在，跳过。" % p.name)
    print("共释放 %s。" % human(freed))


步骤 4：体检与清理
原始语料（全量扫描，比较慢）：


  pretrain(raw): 行数=1270238  解析失败=0
  pretrain(raw): n=1270238
    长度  min=5  p25=399  p50=753  p75=1173  p90=1455  p95=1988  p99=5930  max=28852


    保留比例 <=128:1.5%  <=256:11.4%  <=384:23.6%  <=512:34.3%  <=768:51.0%  <=1024:66.0%  <=2048:95.2%  <=4096:96.5%


  nana(raw): 行数=110216  解析失败=0
  nana(raw): n=110216
    长度  min=10  p25=301  p50=393  p75=515  p90=679  p95=832  p99=1470  max=12638
    保留比例 <=128:0.2%  <=256:14.0%  <=384:47.9%  <=512:74.7%  <=768:93.5%  <=1024:97.4%  <=2048:99.5%  <=4096:99.9%

清洗产物：


  nana(clean): n=110030  总字节=43.96 MB
  nana(clean): n=110030
    长度  min=59  p25=272  p50=363  p75=483  p90=644  p95=794  p99=1409  max=9892
    保留比例 <=128:0.8%  <=256:20.8%  <=384:55.5%  <=512:79.0%  <=768:94.5%  <=1024:97.7%  <=2048:99.6%  <=4096:99.9%


  pretrain(clean): n=1270238  总字节=1.14 GB
  pretrain(clean): n=1270238
    长度  min=5  p25=399  p50=753  p75=1173  p90=1455  p95=1988  p99=5930  max=28852


    保留比例 <=128:1.5%  <=256:11.4%  <=384:23.6%  <=512:34.3%  <=768:51.0%  <=1024:66.0%  <=2048:95.2%  <=4096:96.5%

总览：
  [有] pretrain_t2t_mini.jsonl            1.16 GB
  [有] nana_catgirl_dataset_110k.jsonl    206.74 MB
  [有] pretrain_clean.jsonl               1.16 GB
  [有] nana_clean.jsonl                   46.22 MB

样本：
  --- nana_clean.jsonl ---
  user:能否详细说明一下《马伯里诉麦迪逊案》对美国司法界的重要影响？\nbot:主人问得好难喵...\n这个案子就是马布里没当上法官\n告到最高法院\n然后法院说自己有权说法律违不违宪\n这样法院就能管立法和行政了\n司法也独立起来\n不用听别人指挥捏\n后来好多国家都学这个\n主人听懂了没\n不懂可以再问奈奈喵
  --- pretrain_clean.jsonl ---
  给我生成一首有关秋天的诗歌。秋日早晨，清风拂面。\n金色的叶子，似火在燃烧。\n露珠晶莹，如珍珠般美丽。\n秋的气息，弥漫在空气中。\n余音袅袅，如鸟儿的歌唱。\n美丽的秋天，是大自然的馈赠。帮我想一些创意，给即将到来的公司年会准备节目。一些节目比如能否请一位表演者为我们表演一曲钢琴曲，或者请一位小提琴手为大家演奏一首古典曲目。如果想要画面更具有视觉冲击力，可以安排一个魔术师或者杂技演员的表演。另外，也可以

清理中间产物：
  开关为 False，跳过清理。
